# Predicao - Credit Risk Intelligence Platform

Este notebook consome os modelos registrados no MLflow/Unity Catalog (notebook `18_mlflow`) e gera previsoes de probabilidade de inadimplencia sobre `credit_risk.gold.ml_test`. NAO retreina modelos, NAO faz tuning, NAO usa dados de treinamento para gerar previsoes.

**Dataset**: `credit_risk.gold.ml_test` (48.744 clientes, 229 features)
**Modelos**: Dummy, Logistic Regression, Random Forest, XGBoost, LightGBM
**Output**: `credit_risk.analytics.predictions` (append-only)

In [0]:
# ============================================================
# INSTALACAO DE BIBLIOTECAS (XGBoost e LightGBM)
# Necessario para carregar os modelos registrados que usam esses frameworks
# ============================================================
%pip install xgboost lightgbm -q

In [0]:
# ============================================================
# SECAO 1 - CONFIGURACAO INICIAL
# ============================================================
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
import time
import uuid
import warnings
from datetime import datetime

from pyspark.sql import functions as F
from pyspark.sql.types import *

# Configuracoes globais
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Configuracoes do projeto
TEST_TABLE = "credit_risk.gold.ml_test"
REGISTRY_TABLE = "credit_risk.analytics.mlflow_model_registry"
PREDICTIONS_TABLE = "credit_risk.analytics.predictions"
AUDIT_TABLE = "credit_risk.analytics.audit_prediction"

# ID de execucao unico para auditoria
EXECUTION_ID = str(uuid.uuid4())
START_TIME = time.time()
START_TIMESTAMP = datetime.now()

# Configurar registry URI para Unity Catalog
mlflow.set_registry_uri("databricks-uc")

print(f"{'='*60}")
print(f"19_PREDICTION - CREDIT RISK INTELLIGENCE PLATFORM")
print(f"Execution ID: {EXECUTION_ID}")
print(f"Start Time: {START_TIMESTAMP}")
print(f"{'='*60}")

In [0]:
# ============================================================
# SECAO 2 - CARREGAR REGISTRO DOS MODELOS
# ============================================================
print("=" * 60)
print("SECAO 2 - Carregando registro dos modelos")
print("=" * 60)

# Carregar tabela de registro do MLflow
# Utilizar a versao mais recente registrada (nao assumir versao 1)
df_registry_all = spark.table(REGISTRY_TABLE).toPandas()

# Selecionar somente modelos com status SUCCESS
df_registry_success = df_registry_all[df_registry_all['status'] == 'SUCCESS'].copy()

# Se houver multiplas execucoes, manter a mais recente por modelo
# (agrupar por model_name e pegar o ultimo created_at)
df_registry_success = df_registry_success.sort_values('created_at')
df_registry_success = df_registry_success.drop_duplicates(subset=['model_name'], keep='last')

print(f"Total de registros no registry: {len(df_registry_all)}")
print(f"Modelos com status SUCCESS (mais recentes): {len(df_registry_success)}")

# Exibir modelos que serao utilizados
print(f"\n--- Modelos para Inferencia ---")
for _, row in df_registry_success.iterrows():
    print(f"  {row['model_type']}")
    print(f"    model_name: {row['model_name']}")
    print(f"    model_version: {row['model_version']}")
    print(f"    run_id: {row['run_id']}")
    print(f"    model_uri: {row['model_uri']}")
    print(f"    feature_count: {row['feature_count']}")
    print()

if len(df_registry_success) == 0:
    raise ValueError("ERRO: Nenhum modelo com status SUCCESS encontrado no registry")

In [0]:
# ============================================================
# SECAO 3 - CARREGAR ML_TEST
# ============================================================
print("=" * 60)
print("SECAO 3 - Carregando ml_test")
print("=" * 60)

# Carregar ml_test
df_test = spark.table(TEST_TABLE).toPandas()
print(f"ml_test: {df_test.shape[0]:,} registros, {df_test.shape[1]} colunas")

# Validacoes
print(f"\n--- Validacoes ---")

# Verificar SK_ID_CURR
has_sk_id = 'SK_ID_CURR' in df_test.columns
print(f"SK_ID_CURR presente: {'SIM' if has_sk_id else 'NAO'}")
if not has_sk_id:
    raise ValueError("ERRO: SK_ID_CURR nao encontrado em ml_test")

# Verificar TARGET - NAO deve existir (prevenir leakage)
has_target = 'TARGET' in df_test.columns
print(f"TARGET presente: {'SIM' if has_target else 'NAO (esperado)'}")
if has_target:
    raise ValueError("ERRO FATAL: TARGET encontrado em ml_test! Possivel data leakage. Execucao interrompida.")

# Verificar duplicatas em SK_ID_CURR
dup_count = df_test['SK_ID_CURR'].duplicated().sum()
print(f"SK_ID_CURR duplicados: {dup_count}")

# Verificar nulos em SK_ID_CURR
null_count = df_test['SK_ID_CURR'].isnull().sum()
print(f"SK_ID_CURR nulos: {null_count}")

# Contar features (excluindo SK_ID_CURR)
feature_cols_test = [c for c in df_test.columns if c != 'SK_ID_CURR']
print(f"Features (excluindo SK_ID_CURR): {len(feature_cols_test)}")

# Verificar tipos de dados
dtype_counts = df_test.dtypes.value_counts()
print(f"\n--- Tipos de Dados ---")
for dtype, count in dtype_counts.items():
    print(f"  {dtype}: {count} colunas")

# Colunas categoricas
categorical_cols = df_test.select_dtypes(include=['object']).columns.tolist()
print(f"\nColunas categoricas (string): {len(categorical_cols)}")

# Verificar nulos
total_nulls = df_test[feature_cols_test].isnull().sum().sum()
print(f"Total de valores nulos: {total_nulls:,}")

print("\nValidacoes concluidas com sucesso!")

In [0]:
# ============================================================
# SECAO 4 - PREPARACAO DAS FEATURES
# ============================================================
print("=" * 60)
print("SECAO 4 - Preparacao das features")
print("=" * 60)

# Separar SK_ID_CURR e features
id_column = "SK_ID_CURR"
feature_columns = [c for c in df_test.columns if c != id_column]

X_test = df_test[feature_columns].copy()
sk_ids = df_test[id_column].copy()

print(f"X_test (features): {X_test.shape}")
print(f"SK_ID_CURR: {sk_ids.shape}")
print(f"Total de features: {len(feature_columns)}")

# Validar que sao 229 features (como esperado pelos modelos)
expected_feature_count = 229
if len(feature_columns) != expected_feature_count:
    print(f"WARNING: Esperado {expected_feature_count} features, encontrado {len(feature_columns)}")
else:
    print(f"OK: {expected_feature_count} features confirmadas")

# IMPORTANTE: NAO realizar novas transformacoes de negocio
# O dataset ml_test ja foi preparado pelo notebook 14_gold_dataset_ml
# NAO recalcular imputacao, NAO normalizar, NAO alterar categorizacoes
print("\nFeatures preparadas (sem transformacoes adicionais)")

In [0]:
# ============================================================
# SECAO 5 - VALIDAR FEATURE SCHEMA
# ============================================================
print("=" * 60)
print("SECAO 5 - Validando feature schema contra assinatura MLflow")
print("=" * 60)

schema_validation = []

for _, row in df_registry_success.iterrows():
    model_name = row['model_type']
    model_uri = row['model_uri']

    print(f"\n--- Validando: {model_name} ---")

    try:
        # Carregar informacoes do modelo (assinatura)
        model_info = mlflow.models.get_model_info(model_uri)
        signature = model_info.signature

        if signature is not None and signature.inputs is not None:
            expected_features = [inp.name for inp in signature.inputs.inputs]
            expected_count = len(expected_features)
        else:
            expected_features = []
            expected_count = 0
            print(f"  WARNING: Assinatura sem inputs definidos")

        actual_features = feature_columns
        actual_count = len(actual_features)

        # Comparar contagem, nomes e ordem
        count_match = expected_count == actual_count
        names_match = expected_features == actual_features
        schema_match = count_match and names_match

        result = {
            'MODEL': model_name,
            'EXPECTED_FEATURES': expected_count,
            'ACTUAL_FEATURES': actual_count,
            'SCHEMA_MATCH': 'MATCH' if schema_match else 'MISMATCH',
            'STATUS': 'PASS' if schema_match else 'FAIL',
        }

        if schema_match:
            print(f"  EXPECTED: {expected_count} features | ACTUAL: {actual_count} features")
            print(f"  SCHEMA_MATCH: MATCH")
            print(f"  STATUS: PASS")
        else:
            print(f"  EXPECTED: {expected_count} features | ACTUAL: {actual_count} features")
            if not count_match:
                print(f"  MISMATCH: Contagem diferente")
            if not names_match:
                missing = set(expected_features) - set(actual_features)
                extra = set(actual_features) - set(expected_features)
                if missing:
                    print(f"  Features faltando: {list(missing)[:5]}")
                if extra:
                    print(f"  Features extras: {list(extra)[:5]}")
                if set(expected_features) == set(actual_features) and expected_features != actual_features:
                    print(f"  MISMATCH: Ordem diferente")
            print(f"  STATUS: FAIL")

    except Exception as e:
        print(f"  ERRO: {str(e)}")
        result = {
            'MODEL': model_name,
            'EXPECTED_FEATURES': 0,
            'ACTUAL_FEATURES': len(feature_columns),
            'SCHEMA_MATCH': 'ERROR',
            'STATUS': 'FAIL',
        }

    schema_validation.append(result)

df_schema_validation = pd.DataFrame(schema_validation)
print(f"\n--- Resumo da Validacao de Schema ---")
display(df_schema_validation)

# Filtrar modelos que passaram na validacao
valid_indices = [i for i, v in enumerate(schema_validation) if v['STATUS'] == 'PASS']
valid_models = df_registry_success.iloc[valid_indices].copy()
print(f"\nModelos validados para inferencia: {len(valid_models)}")
if len(valid_models) == 0:
    raise ValueError("ERRO: Nenhum modelo passou na validacao de schema")

In [0]:
# ============================================================
# SECAO 6-7 - CARREGAR MODELOS E GERAR PREDICOES
# ============================================================
print("=" * 60)
print("SECAO 6-7 - Carregando modelos e gerando predicoes")
print("=" * 60)

all_predictions = []
model_load_info = []

for _, row in valid_models.iterrows():
    model_name = row['model_type']
    model_uri = row['model_uri']
    model_version = row['model_version']
    run_id = row['run_id']
    registry_name = row['model_name']

    print(f"\n--- Processando: {model_name} ---")
    print(f"  URI: {model_uri}")

    try:
        # Carregar modelo como sklearn Pipeline
        # Todos os modelos foram logados como sklearn Pipelines (preprocessor + classifier)
        # Usar mlflow.sklearn.load_model para acessar predict_proba()
        model = mlflow.sklearn.load_model(model_uri)
        print(f"  Modelo carregado: PASS")

        # Gerar predicoes (class labels: 0 ou 1)
        predictions = model.predict(X_test)
        print(f"  predict(): {len(predictions)} predicoes geradas")

        # Gerar probabilidades da classe positiva P(TARGET=1)
        # sklearn Pipeline possui predict_proba()
        proba = model.predict_proba(X_test)[:, 1]
        print(f"  predict_proba(): {len(proba)} probabilidades geradas")

        # Estatisticas
        print(f"  Probabilidade - min: {proba.min():.4f} | max: {proba.max():.4f} | mean: {proba.mean():.4f}")
        print(f"  Predicoes 0: {(predictions == 0).sum():,} | 1: {(predictions == 1).sum():,}")

        all_predictions.append({
            'model_name': model_name,
            'registry_name': registry_name,
            'model_version': model_version,
            'run_id': run_id,
            'predictions': predictions,
            'probabilities': proba,
        })

        model_load_info.append({
            'model_name': model_name,
            'model_version': model_version,
            'run_id': run_id,
            'model_uri': model_uri,
            'load_status': 'PASS',
        })

    except Exception as e:
        print(f"  ERRO: {str(e)}")
        model_load_info.append({
            'model_name': model_name,
            'model_version': model_version,
            'run_id': run_id,
            'model_uri': model_uri,
            'load_status': f'FAIL: {str(e)}',
        })

print(f"\nTotal de modelos processados com sucesso: {len(all_predictions)}")

In [0]:
# ============================================================
# SECAO 8 - ESTRUTURA DA PREDICAO
# ============================================================
print("=" * 60)
print("SECAO 8 - Construindo DataFrame de predicoes")
print("=" * 60)

prediction_records = []

for pred_info in all_predictions:
    model_name = pred_info['model_name']
    registry_name = pred_info['registry_name']
    model_version = pred_info['model_version']
    run_id = pred_info['run_id']
    predictions = pred_info['predictions']
    probabilities = pred_info['probabilities']

    for i in range(len(sk_ids)):
        prediction_records.append({
            'SK_ID_CURR': int(sk_ids.iloc[i]),
            'MODEL_NAME': registry_name,
            'MODEL_TYPE': model_name,
            'MODEL_VERSION': str(model_version),
            'RUN_ID': run_id,
            'PREDICTION': int(predictions[i]),
            'RISK_PROBABILITY': float(probabilities[i]),
            'PREDICTION_TIMESTAMP': START_TIMESTAMP.strftime('%Y-%m-%d %H:%M:%S'),
            'EXECUTION_ID': EXECUTION_ID,
        })

df_predictions = pd.DataFrame(prediction_records)
print(f"Total de predicoes: {len(df_predictions):,}")
print(f"Modelos: {df_predictions['MODEL_TYPE'].nunique()}")
print(f"Clientes: {df_predictions['SK_ID_CURR'].nunique():,}")
print(f"Predicoes por modelo: {len(df_predictions) // df_predictions['MODEL_TYPE'].nunique()}")

print(f"\n--- Amostra de Predicoes ---")
display(df_predictions.head(10))

In [0]:
# ============================================================
# SECAO 9-10 - CLASSIFICACAO DE RISCO E SCORE
# ============================================================
print("=" * 60)
print("SECAO 9-10 - Classificacao de risco e score")
print("=" * 60)

# Thresholds documentados (classificacao operacional inicial)
# LOW:    probability < 0.10
# MEDIUM: 0.10 <= probability < 0.30
# HIGH:   probability >= 0.30
# IMPORTANTE: Essas faixas NAO representam uma decisao de credito real

def classify_risk(prob):
    if prob < 0.10:
        return 'LOW'
    elif prob < 0.30:
        return 'MEDIUM'
    else:
        return 'HIGH'

# Aplicar classificacao
df_predictions['RISK_CATEGORY'] = df_predictions['RISK_PROBABILITY'].apply(classify_risk)

# Score de risco normalizado (0 a 100)
# RISK_SCORE = RISK_PROBABILITY * 100
df_predictions['RISK_SCORE'] = (df_predictions['RISK_PROBABILITY'] * 100).round(2)

# Estatisticas por modelo e categoria
print("\n--- Distribuicao de Categorias de Risco por Modelo ---")
risk_dist = df_predictions.groupby(['MODEL_TYPE', 'RISK_CATEGORY']).size().unstack(fill_value=0)
display(risk_dist)

print("\n--- Estatisticas de Probabilidade por Modelo ---")
prob_stats = df_predictions.groupby('MODEL_TYPE')['RISK_PROBABILITY'].describe()
display(prob_stats)

# Metadados dos thresholds
print("\n--- Thresholds Utilizados ---")
print(f"  LOW:    probability < 0.10")
print(f"  MEDIUM: 0.10 <= probability < 0.30")
print(f"  HIGH:   probability >= 0.30")
print(f"  RISK_SCORE = RISK_PROBABILITY * 100")
print(f"  NOTA: Esses thresholds sao uma classificacao operacional inicial e NAO representam uma decisao de credito real.")

In [0]:
# ============================================================
# SECAO 11 - SALVAR TABELA DE PREDICOES
# ============================================================
print("=" * 60)
print("SECAO 11 - Salvando tabela credit_risk.analytics.predictions")
print("=" * 60)

# Salvar como tabela Delta (append-only)
spark.createDataFrame(df_predictions).write.format("delta").mode("append").saveAsTable(PREDICTIONS_TABLE)

print(f"Registros inseridos: {len(df_predictions):,}")
print(f"Tabela: {PREDICTIONS_TABLE}")

# Estatisticas finais
print(f"\n--- Resumo ---")
print(f"Total de predicoes: {len(df_predictions):,}")
print(f"Modelos: {df_predictions['MODEL_NAME'].nunique()}")
print(f"Clientes unicos: {df_predictions['SK_ID_CURR'].nunique():,}")
print(f"Categorias de risco: {df_predictions['RISK_CATEGORY'].value_counts().to_dict()}")

In [0]:
# ============================================================
# SECAO 12 - AUDITORIA
# ============================================================
print("=" * 60)
print("SECAO 12 - Criando tabela credit_risk.analytics.audit_prediction")
print("=" * 60)

execution_time = time.time() - START_TIME
models_processed = len(all_predictions)
models_failed = len(model_load_info) - models_processed

if models_failed > 0:
    status = 'WARNING' if models_processed > 0 else 'FAIL'
    notes = f"{models_processed} modelos processados, {models_failed} falharam"
else:
    status = 'SUCCESS'
    notes = f"Todos os {models_processed} modelos processados com sucesso"

audit_record = pd.DataFrame([{
    'execution_id': EXECUTION_ID,
    'execution_timestamp': START_TIMESTAMP.strftime('%Y-%m-%d %H:%M:%S'),
    'models_processed': models_processed,
    'models_failed': models_failed,
    'total_predictions': len(df_predictions),
    'test_rows': len(df_test),
    'feature_count': len(feature_columns),
    'status': status,
    'execution_time_seconds': round(execution_time, 1),
    'notes': notes,
}])

# Append-only (nao sobrescreve historico)
spark.createDataFrame(audit_record).write.format("delta").mode("append").saveAsTable(AUDIT_TABLE)

print(f"Status: {status}")
print(f"Models processed: {models_processed}")
print(f"Models failed: {models_failed}")
print(f"Total predictions: {len(df_predictions):,}")
print(f"Test rows: {len(df_test):,}")
print(f"Execution time: {execution_time:.1f}s")

In [0]:
# ============================================================
# SECAO 13 - RESUMO DAS PREDICOES
# ============================================================
print("=" * 60)
print("SECAO 13 - Resumo das predicoes")
print("=" * 60)

# Resumo por modelo
print("\n--- Resumo por Modelo ---")
for pred_info in all_predictions:
    model_name = pred_info['model_name']
    proba = pred_info['probabilities']
    preds = pred_info['predictions']

    n_low = (proba < 0.10).sum()
    n_medium = ((proba >= 0.10) & (proba < 0.30)).sum()
    n_high = (proba >= 0.30).sum()

    print(f"\n{model_name} (version {pred_info['model_version']})")
    print(f"  Predicoes 0: {(preds == 0).sum():,} | 1: {(preds == 1).sum():,}")
    print(f"  Probabilidade: min={proba.min():.4f} | max={proba.max():.4f} | mean={proba.mean():.4f}")
    print(f"  Risco: LOW={n_low:,} | MEDIUM={n_medium:,} | HIGH={n_high:,}")

In [0]:
# ============================================================
# SECAO 14 - RESUMO EXECUTIVO
# ============================================================
execution_time_final = time.time() - START_TIME

print("=" * 50)
print("19_PREDICTION - SUMMARY")
print("=" * 50)
print()
print(f"Execution ID: {EXECUTION_ID}")
print(f"Timestamp: {START_TIMESTAMP}")
print()
print(f"Test rows: {len(df_test):,}")
print(f"Features: {len(feature_columns)}")
print(f"SK_ID_CURR: {sk_ids.nunique():,} clientes unicos")
print(f"TARGET presente: NAO (validado)")
print()
print(f"Models processed: {models_processed}")
print(f"Models failed: {models_failed}")
print()
print("Modelos utilizados:")
for pred_info in all_predictions:
    print(f"  - {pred_info['registry_name']} (version {pred_info['model_version']})")
print()
print(f"Total predictions: {len(df_predictions):,}")
print()
print("Risk distribution:")
risk_counts = df_predictions['RISK_CATEGORY'].value_counts()
for cat in ['LOW', 'MEDIUM', 'HIGH']:
    count = risk_counts.get(cat, 0)
    pct = count / len(df_predictions) * 100
    print(f"  {cat}: {count:,} ({pct:.1f}%)")
print()
print(f"Predictions table: {PREDICTIONS_TABLE}")
print(f"Audit table: {AUDIT_TABLE}")
print()
print(f"Execution time: {execution_time_final:.1f}s ({execution_time_final/60:.1f} min)")
print()
print(f"Status: {status}")
print("=" * 50)